# HizmetNabiz validation companion

## tl;dr

The fixed official-data cohort contains **90,565** unique requests. All five blocking quality checks and all independent reconciliation checks pass. Median resolution is **11.47 hours** and P90 is **233.86 hours**. Due-date coverage is only **0.34%**, so the on-time headline is suppressed. `15 BROOKLYN` is the first investigation candidate under the explicit triage index; this is observational, not a causal fairness conclusion.

## Context & Methods

This audit notebook reads only committed analytical artifacts and the compact source manifest. It independently checks reconciliation and surfaces a bounded priority table for review.

### Key Assumptions

- The cohort is requests created from 1 January through 7 January 2025.
- Duration is descriptive administrative latency, not resident-perceived resolution.
- Peer adjustment uses agency x complaint type, with documented sparse fallbacks.
- The scenario's 25% reduction is hypothetical and non-causal.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
summary = json.loads((ROOT / 'data/processed/analysis_summary.json').read_text())
quality = json.loads((ROOT / 'data/processed/quality_report.json').read_text())
receipt = json.loads((ROOT / 'reports/validation_receipt.json').read_text())
boards = pd.read_csv(ROOT / 'data/processed/board_metrics.csv')
scenario = pd.read_csv(ROOT / 'data/processed/capacity_scenario.csv')

## Data

### 1. Inspect source and quality coverage

In [2]:
pd.DataFrame(quality['checks'])[[
    'name', 'value', 'operator', 'threshold', 'passed'
]]

                       name         value operator  threshold  passed
0                 row_count  90565.000000       >=  50000.000    True
1        duplicate_key_rate      0.000000       <=      0.000    True
2   created_date_parse_rate      1.000000       >=      0.999    True
3  community_board_coverage      0.992790       >=      0.850    True
4    negative_duration_rate      0.000022       <=      0.001    True

## Results

### 2. Verify headline metrics

In [3]:
pd.Series({
    'requests': summary['city']['requests'],
    'closed_rate': summary['city']['closed_rate'],
    'median_resolution_hours': summary['city']['median_resolution_hours'],
    'p90_resolution_hours': summary['city']['p90_resolution_hours'],
    'due_date_coverage': summary['city']['due_date_coverage'],
    'on_time_decision_ready': summary['city']['on_time_rate_decision_ready'],
})

requests                        90565
closed_rate                  0.993022
median_resolution_hours     11.466667
p90_resolution_hours       233.863333
due_date_coverage            0.003401
on_time_decision_ready          False
dtype: object

### 3. Review the reliability-adjusted investigation shortlist

In [4]:
boards[[
    'community_board', 'borough', 'requests', 'delay_index',
    'high_delay_rate_eb', 'high_delay_ci_low', 'high_delay_ci_high',
    'excess_delay_hours'
]].head(8).round(3)

  community_board    borough  ...  high_delay_ci_high  excess_delay_hours
0     15 BROOKLYN   BROOKLYN  ...               0.405          120585.625
1        12 BRONX      BRONX  ...               0.310          428249.484
2       04 QUEENS     QUEENS  ...               0.424          204889.073
3    12 MANHATTAN  MANHATTAN  ...               0.366          383664.119
4       08 QUEENS     QUEENS  ...               0.309          163415.572
5        11 BRONX      BRONX  ...               0.321           81423.036
6    10 MANHATTAN  MANHATTAN  ...               0.305          483018.034
7       10 QUEENS     QUEENS  ...               0.313          140556.607

[8 rows x 8 columns]

### 4. Re-run artifact invariants

In [5]:
assert receipt['passed'] is True
assert all(receipt['checks'].values())
assert summary['city']['requests'] == 90_565
assert summary['city']['on_time_rate'] is None
assert scenario['allocated_cases'].sum() <= 500
assert scenario['allocated_cases'].max() <= 125
{
    'reconciliation_checks': len(receipt['checks']),
    'all_passed': all(receipt['checks'].values()),
    'scenario_allocated_cases': int(scenario['allocated_cases'].sum()),
}

{'reconciliation_checks': 12, 'all_passed': True, 'scenario_allocated_cases': 500}

## Takeaways

- The cohort and published artifacts reconcile at 90,565 requests.
- Long-tail latency is material: P90 is about 20 times the median, so a mean-only dashboard would be inadequate.
- The due-date metric fails its coverage guardrail and is correctly suppressed.
- The priority table is a disciplined investigation queue, not evidence that any board or agency caused unequal outcomes.
- A longer, seasonally stratified cohort and population context are required before operational deployment.